# NumPy 1 — arrays, creation, dtypes, shapes

Every section starts with a tiny array you can check by eye. The real data
(`../data/hourly_power_clean.csv`) appears only at the end of a section, in one short cell.

**What's in here**
- creating arrays: `array`, `arange`, `linspace`, `zeros`, `ones`, `full`, `eye`
- random numbers with `default_rng`
- inspecting: `shape`, `ndim`, `size`, `dtype`
- `astype` and the dtype traps: truncation, NaN, overflow, float32, division
- `reshape`, `ravel`, `.T`, `np.newaxis`
- views vs copies (the silent bug)
- stacking arrays

In [1]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 120)

## 1. Creating arrays

`np.array` turns a Python list into an array.

In [2]:
a = np.array([1, 2, 3])
a

array([1, 2, 3])

In [3]:
print("dtype:", a.dtype)
print("shape:", a.shape)

dtype: int64
shape: (3,)


`shape` is `(3,)`: a 1-D array with 3 elements. `int64`: all values are integers.

A nested list gives a 2-D array (rows, columns).

In [4]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])
m

array([[1, 2, 3],
       [4, 5, 6]])

In [5]:
print("shape:", m.shape)     # 2 rows, 3 columns
print("ndim :", m.ndim)
print("size :", m.size)      # total number of elements

shape: (2, 3)
ndim : 2
size : 6


NumPy picks **one** dtype for the whole array. One float among ints makes everything float.

In [6]:
b = np.array([1, 2.5, 3])
print(b)
print(b.dtype)

[1.  2.5 3. ]
float64


**Pitfall:** one string turns everything into strings. `1 + 1` is then impossible.

In [7]:
s = np.array([1, "two", 3.0])
print(s)
print(s.dtype)

['1' 'two' '3.0']
<U32


### `arange` and `linspace`

`arange(start, stop, step)` excludes the stop, like Python's `range`.

In [8]:
np.arange(5)

array([0, 1, 2, 3, 4])

In [9]:
np.arange(2, 10, 2)

array([2, 4, 6, 8])

`linspace(start, stop, num)` includes the stop and you say **how many** points you want.

In [10]:
np.linspace(0, 1, 5)

array([0.  , 0.25, 0.5 , 0.75, 1.  ])

**Pitfall:** `arange` with a float step can give one element more or less than you expect
because of rounding. For float grids use `linspace`.

In [11]:
print(np.arange(0, 1, 0.25))          # 4 elements, stop excluded
print(len(np.arange(0, 1, 0.1)))      # you might expect 10
print(len(np.linspace(0, 1, 11)))     # exactly what you asked for

[0.   0.25 0.5  0.75]
10
11


### `zeros`, `ones`, `full`, `eye`

These take a **shape**. A shape is a tuple: `(2, 3)` means 2 rows, 3 columns.

In [12]:
np.zeros(3)

array([0., 0., 0.])

In [13]:
np.ones((2, 3))

array([[1., 1., 1.],
       [1., 1., 1.]])

In [14]:
np.full((2, 2), np.nan)

array([[nan, nan],
       [nan, nan]])

In [15]:
np.eye(3)

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

`zeros_like(x)` copies the shape **and dtype** of `x`. Handy to pre-allocate a result.

In [16]:
x = np.array([1, 2, 3])
z = np.zeros_like(x)
print(z)
print(z.dtype)         # int, because x is int

[0 0 0]
int64


## 2. Random numbers — `default_rng`

Create one generator with a seed. The same seed always gives the same numbers.

In [17]:
rng = np.random.default_rng(42)
rng.normal(0, 1, size=3)

array([ 0.3047, -1.04  ,  0.7505])

In [18]:
rng.integers(0, 5, size=6)       # 5 is excluded

array([0, 3, 1, 0, 2, 4])

In [19]:
rng.choice(["wind", "solar", "gas"], size=4)

array(['gas', 'gas', 'gas', 'gas'], dtype='<U5')

Two generators with the same seed produce the same sequence.

In [20]:
r1 = np.random.default_rng(7)
r2 = np.random.default_rng(7)
print(r1.normal(size=3))
print(r2.normal(size=3))

[ 0.0012  0.2987 -0.2741]
[ 0.0012  0.2987 -0.2741]


**Interview check:** "Why does your result change every run?" → no seed, or the cells were
run in a different order (every draw advances the generator).

## 3. Inspecting an array

`shape` of a 1-D array is `(n,)`, a tuple with one number. A column vector is `(n, 1)`.
These are different objects; keep the distinction in mind.

In [21]:
v = np.array([10, 20, 30, 40])
print(v)
print("shape:", v.shape)

[10 20 30 40]
shape: (4,)


In [22]:
col = v.reshape(-1, 1)      # -1 means "work it out": 4 rows, 1 column
print(col)
print("shape:", col.shape)

[[10]
 [20]
 [30]
 [40]]
shape: (4, 1)


## 4. `astype` and the dtype traps

`astype` returns a **new** array with another dtype.

Float → int **truncates towards zero**; it does not round.

In [23]:
f = np.array([1.9, -1.9, 2.5])
print(f.astype(int))

[ 1 -1  2]


In [24]:
print(np.round(f))                 # 2, -2, 2  (2.5 rounds to the even number 2)
print(np.round(f).astype(int))

[ 2. -2.  2.]
[ 2 -2  2]


**Pitfall:** `astype(int)` on a NaN gives a nonsense integer, silently.

In [25]:
with np.errstate(invalid="ignore"):
    print(np.array([1.0, np.nan]).astype(int))

[                   1 -9223372036854775808]


### NaN needs a float dtype

There is no integer NaN. Putting NaN into an int array fails.

In [26]:
ints = np.array([1, 2, 3])
try:
    ints[0] = np.nan
except ValueError as e:
    print("ValueError:", e)

ValueError: cannot convert float NaN to integer


In [27]:
floats = ints.astype(float)
floats[0] = np.nan
print(floats)
print(floats.dtype)

[nan  2.  3.]
float64


NaN is not equal to anything, not even itself. Use `np.isnan`.

In [28]:
print(np.nan == np.nan)
print(np.isnan(floats))

False
[ True False False]


### Integer overflow

NumPy integers have a fixed width. `int8` holds −128..127; 100 + 100 wraps around.

In [29]:
small = np.array([100, 100], dtype=np.int8)
with np.errstate(over="ignore"):
    print(small + small)

[-56 -56]


`.sum()` upcasts to int64 by default, so the sum is fine; elementwise `+` is not.

In [30]:
print(small.sum())

200


**Interview check:** "The sum of a positive column is negative." → overflow in a small
int dtype, typically after `dtype=np.int32` on a big volume column.

### float32 vs float64

float32 keeps about 7 significant digits. Adding 1 to 16,777,216 does nothing in float32.

In [31]:
print(np.float32(16_777_216) + np.float32(1))
print(np.float64(16_777_216) + np.float64(1))

16777216.0
16777217.0


Summing many float32 values drifts. One million times 0.1 should be 100,000.

In [32]:
x64 = np.full(1_000_000, 0.1)
x32 = x64.astype(np.float32)
print(x64.sum())
print(x32.sum(dtype=np.float32))

99999.9999999998
100000.086


### Division

`/` always gives floats. `//` is floor division and keeps ints.

In [33]:
a = np.array([7, 8, 9])
print(a / 2)
print(a // 2)

[3.5 4.  4.5]
[3 4 4]


**Pitfall:** division by zero gives `inf` or `nan` with a warning, not an error.
It can slip through silently.

In [34]:
with np.errstate(divide="ignore", invalid="ignore"):
    print(np.array([1.0, 0.0, -1.0]) / 0.0)

[ inf  nan -inf]


## 5. Reshaping

`reshape` rearranges the same 6 values into 2 rows of 3. The total size must match.

In [35]:
m = np.arange(6)
print(m)

[0 1 2 3 4 5]


In [36]:
m.reshape(2, 3)

array([[0, 1, 2],
       [3, 4, 5]])

Values fill **row by row** (row-major order): 0 1 2 go in the first row.

`-1` lets NumPy work out one dimension.

In [37]:
m.reshape(3, -1)

array([[0, 1],
       [2, 3],
       [4, 5]])

`.T` transposes: rows become columns.

In [38]:
m.reshape(2, 3).T

array([[0, 3],
       [1, 4],
       [2, 5]])

`ravel()` flattens back to 1-D.

In [39]:
m.reshape(2, 3).ravel()

array([0, 1, 2, 3, 4, 5])

**Pitfall:** `.T` on a 1-D array does nothing. You need `reshape(-1, 1)` for a column.

In [40]:
print(m.T.shape)
print(m.reshape(-1, 1).T.shape)

(6,)
(1, 6)


`order="F"` fills column by column instead.

In [41]:
print(np.arange(6).reshape(2, 3, order="C"))
print()
print(np.arange(6).reshape(2, 3, order="F"))

[[0 1 2]
 [3 4 5]]

[[0 2 4]
 [1 3 5]]


Real data: 48 hourly values become 2 days × 24 hours, one row per day.

In [42]:
df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
cons = df["consumption_mwh"].to_numpy()
two_days = cons[:48].reshape(2, 24)
print(two_days.shape)
print(two_days[:, :4])          # first 4 hours of each day

(2, 24)
[[26858.4 26177.8 26229.4 25381.3]
 [26785.3 25826.5 25799.8 26081.4]]


## 6. Views vs copies — the silent bug

A basic slice (`a[::2]`, `a[2:5]`) is a **view**: it shares memory with the original.
Writing into the view writes into the original.

In [43]:
a = np.array([0., 1., 2., 3., 4., 5.])
view = a[::2]           # every second element
print("view:", view)

view: [0. 2. 4.]


In [44]:
view[:] = -1
print("view:", view)
print("a   :", a)       # a changed too!

view: [-1. -1. -1.]
a   : [-1.  1. -1.  3. -1.  5.]


Fancy indexing (a list of positions) returns a **copy**. Writing into it leaves the original alone.

In [45]:
a = np.array([0., 1., 2., 3., 4., 5.])
fancy = a[[0, 2, 4]]
fancy[:] = -1
print("fancy:", fancy)
print("a    :", a)      # unchanged

fancy: [-1. -1. -1.]
a    : [0. 1. 2. 3. 4. 5.]


`np.shares_memory` tells you which one you have. `.copy()` always gives an independent array.

In [46]:
print(np.shares_memory(a, a[::2]))
print(np.shares_memory(a, a[[0, 2]]))
print(np.shares_memory(a, a[::2].copy()))

True
False
False


`reshape` also gives a view when it can.

In [47]:
a = np.arange(6)
r = a.reshape(2, 3)
r[0, 0] = 99
print(r)
print(a)

[[99  1  2]
 [ 3  4  5]]
[99  1  2  3  4  5]


Arithmetic like `b * 2` creates a new array. In-place `b *= 2` modifies `b` itself.

In [48]:
b = np.array([1., 2., 3.])
c = b * 2
print("c:", c)
print("b:", b)         # unchanged

c: [2. 4. 6.]
b: [1. 2. 3.]


In [49]:
b *= 2
print("b:", b)         # changed in place

b: [2. 4. 6.]


**Interview check:** "You normalised a slice and the original data changed." → the slice
was a view; take `.copy()` first.

## 7. `np.newaxis` — adding a dimension

`x[:, np.newaxis]` turns `(n,)` into `(n, 1)`. `None` does the same as `np.newaxis`.

In [50]:
x = np.array([1, 2, 3])
print(x.shape)
print(x[:, np.newaxis].shape)
print(x[np.newaxis, :].shape)
print(x[:, None].shape)

(3,)
(3, 1)
(1, 3)
(3, 1)


In [51]:
x[:, np.newaxis]

array([[1],
       [2],
       [3]])

sklearn wants a 2-D `X` of shape (n_samples, n_features), so a single feature needs this.

In [52]:
temp = df["temp_c"].to_numpy()
X = temp[:, np.newaxis]
print(temp.shape, "->", X.shape)

(17520,) -> (17520, 1)


## 8. Stacking arrays

Two small arrays to stack.

In [53]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

`concatenate` joins end to end (along the existing axis).

In [54]:
np.concatenate([a, b])

array([1, 2, 3, 4, 5, 6])

`vstack` puts them as rows.

In [55]:
np.vstack([a, b])

array([[1, 2, 3],
       [4, 5, 6]])

`column_stack` puts them as columns. This is how you build a feature matrix by hand.

In [56]:
np.column_stack([a, b])

array([[1, 4],
       [2, 5],
       [3, 6]])

**Pitfall:** `hstack` on 1-D arrays **appends** them, it does not make columns.

In [57]:
np.hstack([a, b])

array([1, 2, 3, 4, 5, 6])

`stack` creates a new axis; `axis=1` is the same as `column_stack` here.

In [58]:
print(np.stack([a, b]).shape)
print(np.stack([a, b], axis=1).shape)

(2, 3)
(3, 2)


Real data: a design matrix from a constant, temperature and hour. Always print the shape.

In [59]:
hour = df["time"].dt.hour.to_numpy()
X = np.column_stack([np.ones(len(df)), temp, hour])
print(X.shape)
print(X[:3])

(17520, 3)
[[ 1.    0.11  0.  ]
 [ 1.   -0.18  1.  ]
 [ 1.   -1.11  2.  ]]


## Quick reference

| Task | Code |
|---|---|
| float grid | `np.linspace(a, b, n)` |
| reproducible randoms | `rng = np.random.default_rng(42); rng.normal(size=n)` |
| pre-allocate like x | `np.zeros_like(x, dtype=float)` |
| round then cast | `np.round(x).astype(int)` |
| independent slice | `x[i:j].copy()` |
| column vector | `x[:, None]` or `x.reshape(-1, 1)` |
| features to matrix | `np.column_stack([f1, f2, f3])` |
| detect view | `np.shares_memory(a, b)` |